# Stage1 09:46 同业早盘广度对现金相对 SOXX 的增量信息

原 39 项为配对对照，新增一个固定来源坐标；先做无拟合 Qlib 来源、成熟度和训练新颖性预检。

In [1]:
import hashlib,json,os
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
root=Path.cwd().resolve();card=root/'research/stage1_0946_peer_breadth_cash_h5_information_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
parent=plan['parent'];pc=root/parent['source_card']
for artifact,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
 assert sha(pc/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/artifact)==parent[key]
assert json.loads((root/parent['nas_readback']).read_text())['NAS_archive_sha256']==parent['nas_archive_sha256']
src=plan['source'];sc=root/src['card'];ss=json.loads((sc/'summary.json').read_text())
assert sha(sc/'summary.json')==src['summary_sha256'] and ss['recorder_id']==src['recorder_id'] and ss['experiment_id']==src['experiment_id'] and ss['nas_readback']['verified']
assert sha(sc/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts'/src['feature_artifact'])==src['feature_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(pc/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id']);original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features');frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns) and frame.index.equals(base.index) and frame.index.equals(timing.index)
with R.uri_context(str(sc/'qlib_recorder')):
 mapped=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']).load_object(src['feature_artifact'])
assert len(mapped)==2006 and mapped.trade_date.is_unique and mapped.source_known.all()
assert (pd.to_datetime(mapped.assumed_available_at)<pd.to_datetime(mapped.receiver_decision_at)).all()
feature=mapped.set_index('trade_date')['peer_positive_fraction_3'].astype(float).reindex(frame.index).rename(plan['single_mechanism']['feature'])
def h5(series):
 parts=[series.shift(-i) for i in range(5)]
 valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
 return pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=series.index)
label=(-h5(frame.soxx_forward_open_return)).rename('cash_vs_SOXX_H5')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
train_cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC');held_cutoff=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
cohort=plan['cohorts'];train_dates=(frame.index>=pd.Timestamp(cohort['train_signal'][0]))&(frame.index<=pd.Timestamp(cohort['train_signal'][1]))&(mature<train_cutoff)&label.notna();held_dates=(frame.index>=pd.Timestamp(cohort['held_reused_diagnostic'][0]))&(frame.index<=pd.Timestamp(cohort['held_reused_diagnostic'][1]))&(mature<held_cutoff)&label.notna()
assert int(train_dates.sum())>=cohort['minimum_mature_train_rows'] and int(held_dates.sum())>=cohort['minimum_mature_held_rows']
assert feature.loc[train_dates|held_dates].notna().all() and np.isfinite(feature.loc[train_dates|held_dates]).all()
maxcorr=float(base.loc[train_dates,base_names].corrwith(feature.loc[train_dates]).abs().max())
assert maxcorr<plan['single_mechanism']['training_only_max_abs_pairwise_Pearson_stop_at_or_above']
aug=base[base_names].copy();aug[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':aug,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(cohort['train_signal']),'held':tuple(cohort['held_reused_diagnostic'])})
tr=dataset.prepare('train',col_set=['feature','label','meta']);he=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(tr)>=int(train_dates.sum()) and len(he)>=int(held_dates.sum())
report={'status':'conditional_0946_peer_cash_H5_source_maturity_novelty_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'parent_recorder_id':parent['recorder_id'],'source_calendar_rows':len(mapped),'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'train_segment_rows':len(tr),'held_segment_rows':len(he),'max_abs_training_pairwise_Pearson':maxcorr,'candidate_train_unknown_rows':int(feature.loc[train_dates].isna().sum()),'candidate_held_unknown_rows':int(feature.loc[held_dates].isna().sum()),'parent_Recorder_object_loads':4,'source_Recorder_object_loads':1,'DatasetH_builds':1,'model_fits':0,'native_accounts':0,'historical_first_receipt_qualified':False}
(card/'preflight_result.json').write_text(json.dumps(report,indent=2)+'\n')
print({k:report[k] for k in ('status','training_mature_rows','held_mature_rows','max_abs_training_pairwise_Pearson')})

{'status': 'conditional_0946_peer_cash_H5_source_maturity_novelty_preflight_passed_no_fit', 'training_mature_rows': 1055, 'held_mature_rows': 940, 'max_abs_training_pairwise_Pearson': 0.08098169856588952}


In [2]:
import json,os,tempfile
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from qlib.workflow import R
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_settings

addendum=json.loads((card/'pre_fit_rehydration_addendum.json').read_text())
report=json.loads((card/'preflight_result.json').read_text())
assert addendum['frozen_before_first_fit'] and addendum['original_plan_sha256']==sha(card/'plan.json') and addendum['preflight_receipt_sha256']==sha(card/'preflight_result.json')
assert addendum['bootstrap_module_sha256']==sha(root/'src/analysis/seed_disagreement_information.py')
assert report['plan_sha256']==sha(card/'plan.json') and report['model_fits']==report['native_accounts']==0
assert report['training_mature_rows']==int(train_dates.sum()) and report['held_mature_rows']==int(held_dates.sum()) and report['max_abs_training_pairwise_Pearson']==maxcorr
assert not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
tr=tr.loc[tr[('meta','train_eligible')].astype(bool)];he=he.loc[he[('meta','held_eligible')].astype(bool)]
assert len(tr)==report['training_mature_rows'] and len(he)==report['held_mature_rows']
ytr=tr[('label','cash_vs_SOXX_H5')].to_numpy(float);yhe=he[('label','cash_vs_SOXX_H5')].to_numpy(float)
assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
name=plan['single_mechanism']['feature'];inputs={'original39':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),'candidate':(tr['feature'][base_names+[name]].to_numpy(float),he['feature'][base_names+[name]].to_numpy(float))}
assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
state={'status':'fit_attempting','plan_sha256':sha(card/'plan.json'),'addendum_sha256':sha(card/'pre_fit_rehydration_addendum.json'),'fit_attempts':[],'fits_completed':[],'bootstrap_attempts':0,'native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
models={};predictions={};mean=float(ytr.mean());predictions['training_mean']=np.full_like(yhe,mean)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder();state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    dataset.config(dump_all=True,recursive=True);recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,addendum=addendum,dataset=dataset,derived_coordinate=feature)
    for family in ('original39','candidate'):
        state['fit_attempts'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
        model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0));model.fit(inputs[family][0],ytr)
        pred=model.predict(inputs[family][1]);assert np.isfinite(pred).all()
        models[family]=model;predictions[family]=pred
        recorder.save_objects(artifact_path='models',**{family:model});recorder.save_objects(artifact_path='predictions',**{family:pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':pred,'target':yhe})})
        state['fits_completed'].append(family);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');recorder.log_metrics(completed_fits=len(state['fits_completed']))
    dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));scored=pd.DataFrame({'signal_date':dates,'target':yhe,'candidate_prediction':predictions['candidate'],'original39_prediction':predictions['original39'],'training_mean_prediction':predictions['training_mean']})
    calendar=pd.DataFrame({'signal_date':frame.loc[(frame.index>=pd.Timestamp(cohort['held_reused_diagnostic'][0]))&(frame.index<=pd.Timestamp(cohort['held_reused_diagnostic'][1]))].index})
    assert len(calendar)==944 and len(scored)==940
    metrics={'candidate_MSE':float(np.mean((yhe-predictions['candidate'])**2)),'candidate_standardized_coefficient':float(models['candidate'].named_steps['ridge'].coef_[-1]),'train_label_mean':mean,'comparisons':{}}
    all_boot={};all_indices={};all_replicates={};gates={}
    boot=plan['information_gate']['paired_stationary_bootstrap']
    for control in ('original39','training_mean'):
        control_pred=predictions[control]
        col='improvement_vs_'+control
        scored[col]=(scored.target-control_pred)**2-(scored.target-scored.candidate_prediction)**2
        full=calendar.merge(scored[['signal_date',col]],on='signal_date',how='left',validate='one_to_one')
        assert len(full)==944 and int(full[col].notna().sum())==940
        half=[float(g[col].mean()) for g in (scored.loc[scored.signal_date.lt(pd.Timestamp('2025-01-01'))],scored.loc[scored.signal_date.ge(pd.Timestamp('2025-01-01'))])]
        annual=[{'year':int(year),'rows':len(group),'mean_improvement':float(group[col].mean())} for year,group in scored.groupby(scored.signal_date.dt.year,sort=True)]
        complete=[x for x in annual if x['year'] in cohort['complete_held_years']];assert len(complete)==2 and all(x['rows']>=240 for x in complete)
        state['bootstrap_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
        interval,indices,replicates=full_calendar_bootstrap(full[col].to_numpy(float),mean_block=boot['mean_block_sessions'],resamples=boot['draws_each'],seed=boot['seed'],alpha=boot['one_sided_alpha_each'])
        assert interval['valid_replicates']>=9990
        m={'control_MSE':float(np.mean((yhe-control_pred)**2)),'full_mean_improvement':float(scored[col].mean()),'halves':half,'annual':annual,'bootstrap_lower':float(interval['lower_bound']),'bootstrap_upper':float(interval['upper_bound'])}
        metrics['comparisons'][control]=m;all_boot[control]=interval;all_indices[control]=indices;all_replicates[control]=replicates
        gates[control]={'full_positive':m['full_mean_improvement']>0,'both_halves_positive':all(x>0 for x in half),'both_complete_2024_2025_positive':all(x['mean_improvement']>0 for x in complete),'corrected_lower_positive':m['bootstrap_lower']>0}
    gates['coefficient_negative']=metrics['candidate_standardized_coefficient']<0
    passed=all(v for group in gates.values() for v in (group.values() if isinstance(group,dict) else [group]))
    recorder.save_objects(artifact_path='analysis',scored=scored,metrics=metrics,gates=gates,bootstrap=all_boot,bootstrap_indices=all_indices,bootstrap_replicates=all_replicates)
    recorder.log_metrics(candidate_mse=metrics['candidate_MSE'],original39_improvement=metrics['comparisons']['original39']['full_mean_improvement'],training_mean_improvement=metrics['comparisons']['training_mean']['full_mean_improvement'],information_gate_pass=int(passed))
summary={'experiment':plan['experiment'],'decision':'conditional_cash_H5_information_pass_source_PIT_pending_no_account' if passed else 'not_promoted_fixed_0946_peer_breadth_cash_H5_information_failed','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'preflight':'preflight_result.json','information_gate_pass':bool(passed),'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,'new_model_fits':len(state['fits_completed']),'new_bootstraps':state['bootstrap_attempts'],'new_native_accounts':0,'historical_first_receipt_qualified':False,'reused_history_is_independent':False,'recorder_id':state['recorder_id'],'experiment_id':state['experiment_id'],'recorder_uri':str(card/'qlib_recorder'),'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','predictions/*','analysis/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='peer-cash-info-restore-') as temp:
    materialize_nas_evidence(binding,card=Path(temp))
    for name in ('models/original39','models/candidate','analysis/scored'):
        relative=Path('qlib_recorder')/state['experiment_id']/state['recorder_id']/'artifacts'/name
        assert sha(card/relative)==sha(Path(temp)/relative)
summary['nas_evidence']=evidence;summary['nas_scoped_restore_verified']=True;(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state.update(status='completed',nas_archive_sha256=evidence['sha256']);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(summary['decision'],metrics,gates,state['recorder_id'])


not_promoted_fixed_0946_peer_breadth_cash_H5_information_failed {'candidate_MSE': 0.0025892306160339115, 'candidate_standardized_coefficient': -0.000261095054673082, 'train_label_mean': -0.003536702141880777, 'comparisons': {'original39': {'control_MSE': 0.0025894631350398306, 'full_mean_improvement': 2.3251900591877916e-07, 'halves': [2.6234096547186364e-08, 4.689460025044859e-07], 'annual': [{'year': 2023, 'rows': 250, 'mean_improvement': 5.911363668962498e-07}, {'year': 2024, 'rows': 252, 'mean_improvement': -5.341848224499002e-07}, {'year': 2025, 'rows': 250, 'mean_improvement': 4.784946742732035e-09}, {'year': 2026, 'rows': 188, 'mean_improvement': 1.086181448996179e-06}], 'bootstrap_lower': -2.1082716085890565e-06, 'bootstrap_upper': 2.808928955455493e-06}, 'training_mean': {'control_MSE': 0.0023434789844841448, 'full_mean_improvement': -0.00024575163154976637, 'halves': [-0.00025712387995195223, -0.00023271768475091397], 'annual': [{'year': 2023, 'rows': 250, 'mean_improvement':

# Stage1 09:46 同业广度对现金相对 SOXX 的 H5 信息检验

## 中文结论

冻结原 Stage1 39 项输入，只增加 AMD、NVDA、SMH 前 15 分钟同涨比例一个坐标，目标改为未来五会话“现金减 SOXX”开盘对开盘对数收益。此目标针对原账户主要的现金机会损失，区别于已关闭的同一来源对 SOXL 相对收益的 H5 检验。训练期 2018–2022 有 **1055** 个成熟标签，2023–2026 复用评价期有 **940** 个；新增坐标在两段均无缺失，训练期与原 39 项最大绝对相关约 **0.081**。两条 Ridge 各拟合一次，两个对照各做一次预定区块检验，零新账户。

新增标准化系数 **−0.000261**，方向符合“同业普涨降低持有现金价值”的假设，但不够成为稳定信息。候选评价 MSE 为 **0.002589231**：相对原 39 项模型 **0.002589463** 仅改善 **0.000000233**；2024 年改善为负，校正区块下界 **−0.000002108**。相对冻结的训练均值对照 **0.002343479**，候选 MSE **恶化 0.000245752**，前后半段及 2024、2025 年均未改善，校正下界 **−0.000505253**。事前要求同时胜过两个对照的完整、半段、完整年及校正下界门失败。

固定版本决定 `not_promoted_fixed_0946_peer_breadth_cash_H5_information_failed`。不改同涨定义、符号、五日目标、训练年、Ridge 参数或阈值救门，也不运行持仓账户。系统 Python 3.14 的第一次无拟合预检因 OpenMP 初始化 SIGABRT 中止；保留失败记录后，按冻结机械修复计划在仓库 Qlib Python 3.12 环境成功预检与执行，没有因此改变科学合同。

同业 09:45 已完成 bar 在 09:46 决策前可用仍是假设，历史首收与最终版本未核证。这里的 H5 价格误差也不是完整 Stage1 f696 分配器与 b472 行动、风控、恢复链的 0bp 加 IBKR Tiered 自有资金账户。原生 Qlib Recorder `4c278f5781444a79b560d1ecfe22234e` 的模型、预测、完整日历损失和两组区块抽样已归档 NAS 并定向恢复；SOXX 超额、2/5bp 与独立前瞻门均未完成。
